# IAM Users Bulk Edition via Jupyter
Step-by-step guide to simplify bulk edition of [EWC IAM](https://confluence.ecmwf.int/spaces/EWCLOUDKB/pages/439585127/EWC+Identity+and+Access+Management+IAM+Service) users via interactive Jupyter Notebook environments.


## Input

> ✅ The only required column is `email`.

> 💡 Default values for all optional columns are configurable as Jupyter Notebook global parameters. Defaults can also be overwritten on each row.

For simplicity, consider the input example of [users.csv](./users.csv):


email | state | enabled |username | first_name | last_name | roles | comment |
------|-------|---------|---------|------------|-----------|-------|---------|
"john.smith@example.com" | "present" | `true` | | | | | "Adds/updates user with most global defaults (email reused as username)." |
"ada.wong@example.com" | "present" | `false` | | | | | "Adds/updates user with most global defaults but disables login (email reused as username)."  |
"carlos.perez@example.com" | "absent" | | | | | | "Removes user, if exists." |
"philipp.mayer@example.com" | "present" | `true` | "pmayer" | "Philipp"  | "Mayer" | "ewc-jhub-lab-f54924:ewc-jhub-lab-cfe2f3" | "Adds/updates user with optional default overrides for `username`, `first_name`, `last_name` and `roles` (two roles **separated by colon**)."  |


## Usage

Open the [iam-user-bulk-edition-via-jupyter.ipynb]() notebook, start the runtime, and execute cells top to bottom to apply access/permission changes.

## Workflow Stages

1. **Global Parameters**: tenancy name, global default values, `CSV`  path
2. **Dependencies Setup**: fetch dependencies, pin versions, and install
3. **Input Data Loading and Cleaning**: validate and normalize the input `CSV`
4. **Configuration Auto-generation**: generate and preview the equivalent `YAML` configuration changes to be applied on EWC IAM, based on input user `CSV` data
5. **Apply Configuration Changes**: Apply the necessary EWC IAM changes as per the generate `YAML` configuration



----
## Stage 1 - Global Parameters

> ⚠️ Fill in required `tenancy_name` parameter below

> 💡 All other parameters are pre-filled and be left untouched or overriden as needed.


In [ ]:
# Tenancy
tenancy_name: str = ""  # required, no default

# Users
users_csv_path: str = "./users.csv"

# Defaults
deletion_protection: bool = False
state: str = "present"
enabled: bool = True
email_verified: bool = True
initial_login_actions: str = "UPDATE_PASSWORD"
roles: str = "ewc-iam-user"
roles_reconciliation_mode: str = "replace"
first_name: str = "Unknown"
last_name: str = "Unknown"

---
## Stage 2 - Dependencies Setup

In [ ]:
%%bash
dependency_version="1.2.0"
local_source_dir="./ewc-user-tools"

if ! git clone https://github.com/ewcloud/ewc-user-tools.git "${local_source_dir}" 2>/dev/null && [ -d "${local_source_dir}" ] ; then
    echo "Dependency cloning failed! Local source directory ${local_source_dir} already exists."
fi

cd "${local_source_dir}"
git checkout "${dependency_version}"
pip install -r ./items/iam-users-bulk-edition/requirements.txt --no-color
pip install -r ./items/iam-users-bulk-edition/dev-requirements.txt --no-color

---
## Stage 3 - Input Data Loading and Cleaning

### 3.1. Read input user data

In [ ]:
import pandas as pd

df = pd.read_csv(users_csv_path, dtype=str)

if "email" not in df.columns:
    raise ValueError("Column `email` not included in the input CSV. Include the column and values for all rows.")

### 3.2. Drop columns falling outside of the input schema

In [ ]:
REQUIRED_COLUMNS = ["email"]
OVERRIDABLE_COLUMNS = ["username","first_name","last_name","enabled","state","deletion_protection","email_verified","initial_login_actions","roles","roles_reconciliation_mode"]

df = df[df.columns.intersection( REQUIRED_COLUMNS + OVERRIDABLE_COLUMNS )]

### 3.3. De-duplicate Rows
> ⚠️ Duplicated rows are not supported! We keep only the last entry for any given e-mail address.


In [ ]:
dup_email = df["email"].duplicated(keep="last")
df = df[~dup_email]

### 3.4. Trim whitespaces on string fields

In [ ]:
df = df.apply(lambda col: col.str.strip() if pd.api.types.is_string_dtype(col) or pd.api.types.is_object_dtype(col) else col)

### 3.5. Lower-case email addresses

In [ ]:
df["email"] = df["email"].str.lower()

### 3.6. Normalize null values

In [ ]:
null_tokens = {"", "NA", "N/A", "-"} # <- Edit allowed values as needed

def _normalize(value):
    if pd.isna(value):
        return pd.NA
    if str(value).strip().upper() in null_tokens:
        return pd.NA
    return value

df = df.apply(lambda col: col.map(_normalize))

### 3.7. Coerce boolean-like optional columns, if present

In [ ]:
true_tokens = {"true", "yes", "1"}  # <- Edit allowed values as needed
false_tokens = {"false", "no", "0"} # <- Edit allowed values as needed

BOOLEAN_COLUMNS = ["enabled", "deletion_protection", "email_verified"]

def _coerce(value):
    if pd.isna(value):
        return pd.NA
    token = str(value).strip().lower()
    if token in true_tokens:
        return True
    if token in false_tokens:
        return False
    return pd.NA

for col in [c for c in BOOLEAN_COLUMNS if c in df.columns]:
    df[col] = df[col].map(_coerce)

### 3.8. Preview clean input data

In [ ]:
df

---
## Stage 4 - Configuration Auto-generation

### 4.1. Apply defaults

In [ ]:
IN_FIELD_DELIMITER = ":"

def _as_roles(items: list[str]) -> list[dict]:
    return [{"name": name} for name in items]

defaults = {
    "deletion_protection": bool(deletion_protection),
    "state": state,
    "enabled": bool(enabled),
    "email_verified": bool(email_verified),
    "initial_login_actions": [item.strip() for item in str(initial_login_actions).split(IN_FIELD_DELIMITER) if item.strip()],
    "roles": _as_roles([item.strip() for item in str(roles).split(IN_FIELD_DELIMITER) if item.strip()]),
    "roles_reconciliation_mode": roles_reconciliation_mode,
    "first_name": first_name,
    "last_name": last_name,
}

### 4.2. Apply user overrrides

In [ ]:
LIST_FIELDS = ["initial_login_actions", "roles"]

def resolve_user_entry(row: pd.Series) -> dict:
    entry: dict = {"email": row["email"]}
    for field in OVERRIDABLE_COLUMNS:
        value = row.get(field)
        if pd.isna(value):
            continue
        if field in LIST_FIELDS:
            items = [item.strip() for item in str(value).split(IN_FIELD_DELIMITER) if item.strip()] 
            entry[field] = _as_roles(items) if field == "roles" else items
        else:
            entry[field] = value
    return entry

users = [resolve_user_entry(row) for _, row in df.iterrows()]

### 4.3. Assemble the final configuration

In [ ]:
import yaml

CONFIG_YAML_PATH = "./ewc-user-tools/items/iam-users-bulk-edition/vars/inputs.yml"

configuration = {
    "schema_version": 1,
    "tenancy": {
        "name": tenancy_name,
        "users": users,
        "defaults": defaults
        },
    }

if  tenancy_name == "":
    raise ValueError("Global parameter `tenancy_name` is not set. Review 'Stage 1 - Global Parameters' section above")


with open(CONFIG_YAML_PATH, "w") as fp:
    yaml.safe_dump(configuration, fp, sort_keys=False) 

### 4.4. Preview the final configuration

In [ ]:
print(yaml.safe_dump(configuration, indent=2, sort_keys=False))

---
## Stage 5 - Apply Configuration Changes



### 5.1. Resolve path to dependency's executable

In [ ]:
import shutil
import sys
from pathlib import Path

ansible_playbook = Path(sys.executable).with_name("ansible-playbook")
if not ansible_playbook.exists():
    found = shutil.which("ansible-playbook")
    if found is None:
        raise FileNotFoundError("ansible-playbook not found next to the kernel's Python. Check that 'Stage 2 - Dependencies Setup' installed the ansible package into this environment.")
    ansible_playbook = Path(found)

### 5.2. Execute

>⚠️ You will be prompted to enter EWC IAM tenancy admin username and password. This is required by the tooling to make changes on your behalf.

In [ ]:
import os
import getpass
from datetime import datetime
import pexpect

ENTRYPOINT_PATH = f"{os.getcwd()}/ewc-user-tools/items/iam-users-bulk-edition/iam-users-bulk-edition.yml"
USERNAME_RE = r"(?i)username\s*:"
PASSWORD_RE = r"(?i)password\s*:"

cmd = [str(ansible_playbook), ENTRYPOINT_PATH]
child = pexpect.spawn(cmd[0], cmd[1:], encoding="utf-8", timeout=None)

transcript = []
while True:
    idx = child.expect([USERNAME_RE, PASSWORD_RE, pexpect.EOF])
    print(child.before, end="")
    transcript.append(child.before)
    if idx == 0:
        child.sendline(input("IAM Tenant Admin username: "))
    elif idx == 1:
        child.sendline(getpass.getpass("IAM Tenant Admin password: "))
    else:
        break

child.close()

log_path = f"run_{datetime.now():%Y%m%d_%H%M%S}.log"
with open(log_path, "w") as fp:
    fp.write("".join(transcript))
print(f"\nFull output written to {log_path}")

returncode = child.exitstatus if child.exitstatus is not None else -(child.signalstatus or 0)
if returncode != 0:
    raise RuntimeError(f"ansible-playbook exited with code {returncode}; see {log_path}")

print("Subprocess completed successfully.")